# Decoder check - the compiled decoder step against part two's

Notebook 03 stopped at its control on the first run. It used the compiled decoder step, and one of 22 decoder numbers, water IoU on the validation split, came out lower than part two's (0.680 against 0.700, allowed 0.012). Part three's control with part two's step had 0.704. This notebook decides whether the compiled step moves the numbers or the gap was chance, by a rule written down before it runs.

## What runs

- the control's run of notebook 03 (forest, the satellite DINOv3 at layer 21, the three channel input) twice in this session with the new seeds 3 to 8, once with part two's decoder step and once with the compiled step for every seed. The encoder is loaded once, both sides read the same features
- the speed of both steps at 128, 256, 512 and 992 px with the students' feature width, on a synthetic batch, so the gain is known where the decoder minutes are

## The rule, written before the run

Every score column of the decoder rows on val and test, by the seed means, the rule of notebook 03's control. A mean is off when its gap is larger than 0.01 and larger than twice the larger seed SD.

- no mean off. The compiled step agrees, notebook 03 runs its control with part two's step and the students with the measured choice
- one or more off. The compiled step moves the numbers, notebook 03 runs part two's step everywhere
- the compiled side did not run compiled (a compile that failed). Not tested, nothing is decided

The differences per seed (the same seed starts both sides from the same weights and batch order) are printed next to it and do not decide.

## Runtime and time

An A100, the GPU the scoring runs on, since a compiled step is built for the GPU it runs on. Standard RAM is enough, the forest features are about 1 GB. About 15 minutes, an estimate - 12 decoders of about 0.6 minutes, the encoder and the features, the first compile, and the speed check with one compile per size. On a runtime that has the forest data on its disk already (notebook 03's) it is not fetched again.

## Data and the .env file

The forest dataset as notebook 03 takes it, from its private Hugging Face copy when DATASET_HF_OWNER is set, else from its copy on Drive under MyDrive/datasets. HF_TOKEN reads the gated DINOv3 weights and is never printed.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/decoder_check/. The files of the run step for each side (eager/ and compiled/) and decoder_check.json with the verdict, every compared mean and the speeds.

## How to run

Run the cells from top to bottom. A side whose summary is on Drive is not run again. The last cell flushes Drive and then releases the Colab machine (set END_RUNTIME = False in that cell to keep the runtime), and a cell that fails stops the run before it.


In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. Tokens, model cache, GPU check, folders, the forest dataset on the runtime disk.
import json
import os
import time
from pathlib import Path

import pandas as pd
import torch

from sartransfer import env
from sartransfer.distil import score, stage0
from sartransfer.domains import hub, prepared
from sartransfer.domains import run as run_step
from sartransfer.domains.prepared import load_manifest

start = time.perf_counter()
assert torch.cuda.is_available(), "no GPU, pick the A100 runtime and connect again"
DEVICE = "cuda"
MODEL_CACHE = env.model_cache_on_drive()       # before transformers or huggingface_hub load
assert MODEL_CACHE is not None, "Google Drive is not mounted, run the mount cell first"
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the gated DINOv3 weights and the private data copies")
assert HF_TOKEN, "no Hugging Face token was given"
DATA_TOKEN = os.environ.get("HF_WRITE_TOKEN") or HF_TOKEN   # the private dataset copies, as notebook 03
DATASET_OWNER = os.environ.get("DATASET_HF_OWNER", "").strip()   # optional, without it the Drive copy is used

# The control's run of notebook 03 and the rule of its decoder rows (written before the run)
DATASET, ENCODER, MODE = "forty_v1", "dinov3-l-sat", "vv_vh_diff"
LAYER = run_step.PRE_REGISTERED_LAYERS[ENCODER]     # 21, as part two
SEEDS = (3, 4, 5, 6, 7, 8)                          # new seeds, the control ran 0, 1 and 2
TOLERANCE = 0.01                                    # the tie rule of the control
STUDENT_WIDTH = 768                                 # the students' feature width (ViT-B)
SPEED_TILES = (128, 256, 512, 992)                  # the chip sizes of part three's decoder timings

CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
PREP = CONTENT / "sar-transfer-distillation" / "prepared"          # runtime disk, as notebook 03
CACHE = CONTENT / "sar-transfer-distillation" / "compile_cache"    # the compiled steps of this session, as notebook 03
OUT = MYDRIVE / "sar-transfer-distillation" / "decoder_check"      # the results of this notebook
ARMS = {"eager": OUT / "eager", "compiled": OUT / "compiled"}   # eager = part two's step, that of every saved row
ARM_SETTINGS = {"eager": {}, "compiled": {"decoder_path": "compiled", "compile_cache": CACHE}}
RUN = score.run_name(DATASET, ENCODER, LAYER, MODE)
TIMES = {}                       # seconds of every step of this run, by name


def took(since, name):
    """The time of a cell as text, recorded under `name`."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def on_disk():
    """True when the manifest and every split file of the forest dataset are on the runtime disk."""
    if not (PREP / DATASET / "manifest.json").is_file():
        return False
    splits = load_manifest(PREP, DATASET)["splits"]
    return all((PREP / DATASET / file).is_file() for file in prepared.split_files(splits))


if on_disk():
    print(f"{DATASET}  on the runtime disk already")
else:
    clock, source = time.perf_counter(), None
    if DATASET_OWNER:
        try:                                       # first the private Hugging Face copy, every split
            hub.fetch(DATASET, DATASET_OWNER, PREP, token=DATA_TOKEN, splits=None)
            source = "Hugging Face"
        except Exception as err:
            print(f"{DATASET}  not on Hugging Face ({type(err).__name__}, {hide(err)[:200]})")
    if source is None:                             # then the copy on Drive
        copy = MYDRIVE / "datasets" / hub.repo_name(DATASET)
        assert (copy / "manifest.json").is_file(), f"{DATASET} is neither on Hugging Face nor on Drive under {copy}"
        prepared.copy_prepared(copy, PREP, DATASET, splits=None)
        source = "Drive"
    print(f"{DATASET}  from {source} in {(time.perf_counter() - clock) / 60:.1f} min")
man = load_manifest(PREP, DATASET)
print(f"{DATASET}  splits {man['splits']}, {man['chip_size']} px, units {man['units']}")
print(f"GPU     {torch.cuda.get_device_name(0)}")
print(f"results go to   {OUT}")
for folder in (*ARMS.values(), CACHE):
    folder.mkdir(parents=True, exist_ok=True)
print(took(start, "setup"))


In [ ]:
# Runs. The control's run twice with the seeds 3 to 8, part two's step and then the compiled step for every seed
# (run_domain's decoder_path, the speed is measured and printed but does not choose). The encoder is loaded once
# for both sides, so both read the same features. A side whose summary is on Drive is not run again.
start = time.perf_counter()
enc_cache = {}
try:
    for arm, folder in ARMS.items():
        print(f"\n---- {arm}")
        run_step.run_domain(PREP, DATASET, ENCODER, folder, token=HF_TOKEN, layer=LAYER, seeds=SEEDS, mode=MODE,
                            device=DEVICE, enc_cache=enc_cache, **ARM_SETTINGS[arm])
finally:
    enc_cache.clear()                              # the encoder leaves the GPU before the speed check
    torch.cuda.empty_cache()
print(took(start, "runs"))


In [ ]:
# Verdict. The two sides by the control's rule (score.seed_mean_check), every score column of the decoder rows on
# val and test. The words follow the rule written before the run. The differences per seed
# are printed next to it and do not decide. Everything goes to decoder_check.json on Drive.
start = time.perf_counter()
summaries = {arm: pd.read_csv(folder / f"{RUN}__summary.csv") for arm, folder in ARMS.items()}
ran = json.loads((ARMS["compiled"] / f"{RUN}__info.json").read_text(encoding="utf-8"))["times"]
table = score.seed_mean_check(summaries["eager"], summaries["compiled"], TOLERANCE)
print(f"eager = part two's step, compiled = the compiled step, decoder seeds {list(SEEDS)} on both sides")
print("gap = compiled mean - eager mean as a size, allowed = the larger of 0.01 and twice the larger seed SD,")
print("paired_diff = the mean of compiled - eager per seed, seeds_lower = the seeds where compiled is lower")
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
n_off = int(table["off"].sum())
for _, row in table[table["off"]].iterrows():
    print(f"off  {row['column']} {row['split']}, gap {row['gap']:.4f}, allowed {row['allowed']:.4f}")
if ran.get("decoder_path") != "compiled":
    verdict = "not tested"
    words = f"NOT TESTED - the compiled side ran {ran.get('decoder_path')}, see its lines above. Nothing is decided"
elif n_off:
    verdict = "differs"
    words = (f"DIFFERS - {n_off} of {len(table)} means are off. The compiled step moves the numbers, notebook 03 "
             "runs part two's step everywhere")
else:
    verdict = "agrees"
    words = (f"AGREES - none of {len(table)} means is off. Notebook 03 runs its control with part two's step and "
             "the students with the measured choice")
print(f"\n{words}")
REPORT = {"verdict": verdict, "words": words, "means": len(table), "off": n_off, "seeds": list(SEEDS),
          "tolerance": TOLERANCE, "run": RUN, "gpu": torch.cuda.get_device_name(0),
          "compiled_side": {key: value for key, value in ran.items() if key.startswith("decoder")},
          "rows": table.to_dict("records")}
(OUT / "decoder_check.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")
print(f"saved {OUT / 'decoder_check.json'}")
print(took(start, "verdict"))


In [ ]:
# Speed. Both steps at 128, 256, 512 and 992 px with the students' width, batch 8, on a synthetic batch of those
# shapes (run_step.decoder_speed_check, the check every fast run makes). The compiled step is worth it only where it
# saves time, and the largest tiles hold most of the decoder minutes (part three, 992 px, about 21 of the 22 minutes
# of an alpine run). A size whose check fails is printed and the others go on. The speeds are added to
# decoder_check.json.
start = time.perf_counter()
run_step.use_compile_cache(CACHE)
speed = []
for tile in SPEED_TILES:
    try:
        r = run_step.decoder_speed_check(tile, STUDENT_WIDTH, DEVICE)
    except Exception as err:                      # out of memory or anything, the other sizes go on
        r = {"tile": tile, "eager_ms": None, "compiled_ms": None, "error": f"{type(err).__name__}, {hide(err)[:200]}"}
    speed.append(r)
    if r["eager_ms"] is None or r["compiled_ms"] is None:
        eager = "not measured" if r["eager_ms"] is None else f"{r['eager_ms']:.1f} ms"
        print(f"{tile:>4} px  eager {eager}, compiled not measured ({r['error']})")
    else:
        print(f"{tile:>4} px  eager {r['eager_ms']:.1f} ms, compiled {r['compiled_ms']:.1f} ms, "
              f"{100 * (1 - r['compiled_ms'] / r['eager_ms']):.0f} percent less, first compile {r['compile_s']:.0f} s, "
              f"loss gap {r['loss_gap']:.1e} (tolerance {r['loss_tolerance']:.1e}), gradient gap "
              f"{r['grad_gap']:.1e} (tolerance {r['grad_tolerance']:.1e})")
REPORT["speed"] = speed
REPORT["times"] = TIMES
(OUT / "decoder_check.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")
print(f"speeds added to {OUT / 'decoder_check.json'}")
print(took(start, "speed"))


In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
